# Hopfield 统一任务基准

状态：`implementation-complete / not-run`

这是一个可直接上传并在 Colab 运行的单文件实验。它不克隆仓库、不导入旁边的源码文件，也不安装额外依赖。notebook 内联实现 Classical Hopfield、Polynomial/Exponential DAM、Simplicial R12、PSHN 与 Continuous Modern Hopfield，并将共享任务与论文专属机制实验分开。

文件没有保存运行输出。因此，代码完整不等于实验结论已经成立；图下结论只会在 Colab 实际执行后由原始记录生成。

## 0. 组件式实验管线与任务边界

`a 记忆输入 → b 模型存储 → c 检索线索 → d 检索动力学 → e 测量 → f 同图比较`

共享任务复用 `a/c/e/f`，只替换 `b/d`：U1 固定点、U2 噪声恢复、U3 有限容量、U4 吸引域、U5 动力学、U6 虚假吸引子、U7 资源效率。所有模型在 U2/U3/U4 的共同主指标是 **Top-1 memory identification**（终态与哪一条已存记忆最相似），因为它同时适用于二值终态和连续终态。

论文专属机制不混入总排名：Simplicial 做 H1 阶数、H2 随机结构、H3 同参数预算与 H7 结构×相似度；Curved/Explosive 做 H4 状态反馈有效温度和 H5 正反向迟滞；PSHN 做 H6 分组数与 feature-to-prototype。1982 的非对称、遗忘、同步周期和 1985 的有限温度相图/AT 线仍属于各自论文 notebook，避免为同一任务重复造轮子。

In [ ]:
# [环境] 只使用 Colab 预装的 Python/PyTorch/绘图库，不在 notebook 里安装依赖
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from time import perf_counter
from typing import Any, Callable, Iterable
from urllib.request import Request, urlopen
from uuid import uuid4
import json

# [展示] Matplotlib/Pandas 只消费标准结果表，不参与模型更新
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

# [展示] 全局画图风格固定，避免不同实验各自偷偷改视觉编码
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)

# [溯源] 每条原始记录都带 notebook 路径和运行时解析到的 Git 提交
SOURCE_NOTEBOOK = "hopfield-benchmark/hopfield_benchmark_phase1_colab.ipynb"
try:
    # [输入·外部] 只读取 GitHub main 的提交号，不克隆仓库、不下载运行环境
    request = Request(
        "https://api.github.com/repos/Heptazero/nn-labs/commits/main",
        headers={"User-Agent": "hopfield-benchmark-colab"},
    )
    with urlopen(request, timeout=20) as response:
        SOURCE_COMMIT = json.load(response)["sha"]
# [失败边界] 断网不阻止实验，但来源提交明确记为 unresolved，不能静默伪造
except Exception as error:
    SOURCE_COMMIT = "unresolved"
    print(f"warning: could not resolve GitHub commit: {error}")

print(f"torch={torch.__version__}, pandas={pd.__version__}")
print(f"source commit={SOURCE_COMMIT}")

## 1. a / c：共享记忆与共享线索

`a1` 生成独立随机 `{-1,+1}` 模式。`c1` 精确翻转 `round(rho*N)` 个位置；`c2` 生成与记忆独立的随机初态。线索始终在模型循环外生成，同一 trial 的所有模型收到逐元素相同的 tensor。

In [ ]:
# [数据契约] a 组件的输出；模式张量与身份、种子、编码一起传给所有模型
@dataclass(frozen=True)
class MemorySet:
    patterns: torch.Tensor
    pattern_ids: tuple[int, ...]
    dataset_id: str
    data_seed: int
    encoding: str = "binary_pm1"

    @property
    # [观测·数据] patterns:(P,N)，第 0 维就是实际存储模式数
    def P(self) -> int:
        return int(self.patterns.shape[0])

    @property
    # [观测·数据] patterns:(P,N)，第 1 维就是状态空间维数
    def N(self) -> int:
        return int(self.patterns.shape[1])


# [实验控制] 随机数生成器显式局部化；函数不会改写 PyTorch 全局种子
def make_generator(seed: int, device: torch.device | str = "cpu") -> torch.Generator:
    generator = torch.Generator(device=device)
    generator.manual_seed(int(seed))
    return generator


def a1_make_independent_binary(
    N: int,
    P: int,
    data_seed: int,
    *,
    device: torch.device | str = "cpu",
) -> MemorySet:
    # [约束] 空模式集没有可定义的存储或 Top-1 检索任务
    if N <= 0 or P <= 0:
        raise ValueError("N and P must be positive")
    # [输入] bits:(P,N) 独立 Bernoulli(1/2)，先用 int8 节省内存
    bits = torch.randint(
        0,
        2,
        (P, N),
        generator=make_generator(data_seed, device),
        device=device,
        dtype=torch.int8,
    )
    return MemorySet(
        # [更新] {0,1} → {-1,+1}，得到 Hopfield 二值自旋编码
        patterns=bits.mul(2).sub(1),
        pattern_ids=tuple(range(P)),
        dataset_id="independent_binary",
        data_seed=int(data_seed),
    )


def c1_make_hamming_cue(
    target: torch.Tensor,
    corruption_level: float,
    cue_seed: int,
) -> torch.Tensor:
    # [约束] 一条 cue 只能对应一个一维目标状态
    if target.ndim != 1:
        raise ValueError("target must be one-dimensional")
    if not 0.0 <= corruption_level <= 1.0:
        raise ValueError("corruption_level must lie in [0, 1]")
    # [输入] 复制目标，噪声构造不能污染记忆库里的原样本
    cue = target.detach().clone().to(torch.int8)
    # [实验控制] 精确翻转 round(rho*N) 位，不使用期望翻转率近似
    flip_count = int(round(corruption_level * cue.numel()))
    if flip_count:
        # [中介变量] indices:(flip_count,) 无放回抽样，避免同一位翻两次
        indices = torch.randperm(
            cue.numel(),
            generator=make_generator(cue_seed, cue.device),
            device=cue.device,
        )[:flip_count]
        # [更新] 原地取反后，cue 与 target 的汉明距离恰好等于 flip_count
        cue[indices] *= -1
    return cue


# [输入] U6 随机初态；与任何已存记忆独立，用来探测吸引子体积
def c2_make_random_state(N: int, cue_seed: int) -> torch.Tensor:
    if N <= 0:
        raise ValueError("N must be positive")
    bits = torch.randint(
        0, 2, (N,), generator=make_generator(cue_seed), dtype=torch.int8
    )
    return bits.mul(2).sub(1)

## 2. b / d：可替换模型组件

所有适配器实现 `fit → retrieve → resource_summary`，但保留各自原生状态与动力学。

- Classical Hopfield 存储 `W=(1/N)Σ ξξᵀ` 且清零对角线，随机顺序异步更新。
- Polynomial DAM 使用 `E=-Σμ mμ^d`；Exponential DAM 使用负 log-sum-exp 能量代理，二者逐坐标比较 `s_i=+1/-1` 的能量。
- Simplicial R12 使用 `E=-Σσ wσ Sσ`，其中 `wσ=(1/N)Σμ ξσμ`；主适配器以边和三元单形混合，并用异步坐标下降保证可检查的能量轨迹。论文原生同步规则只在专属解释中讨论。
- PSHN 将坐标分成 `k` 组，一次读出时把“其余组相关度的乘积”作为每组记忆系数。
- Continuous Modern Hopfield 使用 `softmax(beta·Xq)X` 返回连续状态，不伪装成二值固定点动力学。

这些能量数值不共享同一零点或尺度，因此 U5 只同图比较公共误差轨迹，不把不同能量直接画成高低排名。

In [ ]:
# [数据契约] d 组件的统一输出；没有定义的轨迹必须留空，不能伪造
@dataclass
class RetrievalResult:
    # [观测·数据] 最终状态可为二值或连续，具体类型写进 diagnostics
    final_state: torch.Tensor
    status: str
    sweeps: int
    state_updates: int
    # [观测·资源] 实现登记的近似操作数，不等同于硬件实测时间
    retrieval_flops: int
    # [观测·数据] 各模型自己的能量标尺；只检查单模型内是否下降
    energy_trace: list[float] = field(default_factory=list)
    error_trace: list[float] = field(default_factory=list)
    diagnostics: dict[str, Any] = field(default_factory=dict)


# [约束] 离散适配器统一从 {-1,+1} 一维状态开始，随后转 float64 防累加溢出
def binary_state(state: torch.Tensor) -> torch.Tensor:
    values = set(torch.unique(state.to(torch.int8)).tolist())
    if state.ndim != 1 or not values.issubset({-1, 1}):
        raise ValueError("state must be one-dimensional and use {-1, +1}")
    return state.detach().clone().to(torch.float64)


# [模型 b/d] 经典二体 Hebb 存储 + 随机顺序异步检索
class ClassicalHopfield:
    model_id = "classical_hebb"

    def fit(self, patterns: torch.Tensor) -> "ClassicalHopfield":
        # [输入] patterns:(P,N)；转 float64 后再做 P 项累加
        binary = patterns.to(torch.float64)
        if binary.ndim != 2:
            raise ValueError("patterns must have shape [P, N]")
        self.N = int(binary.shape[1])
        # [存储] W=(1/N)XᵀX，(N,P)@(P,N)→(N,N)
        self.weights = binary.T @ binary / self.N
        # [约束] W_ii=0，禁止神经元把自身旧值当作局部场证据
        self.weights.fill_diagonal_(0.0)
        return self

    def energy(self, state: torch.Tensor) -> float:
        # [观测·数据] E=-1/2 sᵀWs；只作为异步下降检查，不参与判停
        return float((-0.5 * state @ self.weights @ state).item())

    def retrieve(
        self,
        cue: torch.Tensor,
        *,
        target: torch.Tensor,
        update_seed: int,
        max_sweeps: int,
    ) -> RetrievalResult:
        if max_sweeps <= 0:
            raise ValueError("max_sweeps must be positive")
        # [输入] 复制 cue；后续原地更新不会污染共享线索
        state = binary_state(cue)
        target_f = target.to(torch.float64)
        # [实验控制] 同一 trial 的迭代模型共享 update_seed，消除更新顺序差异
        generator = make_generator(update_seed, state.device)
        # [观测·数据] 先记录 t=0，图中第一点就是损坏线索
        energies = [self.energy(state)]
        errors = [float(torch.mean((state != target_f).to(torch.float64)).item())]
        updates = 0
        first_sweep_unchanged = False
        for sweep in range(1, max_sweeps + 1):
            changed = False
            # [实验控制] 每轮随机排列全部 N 个坐标，避免固定编号顺序偏置
            order = torch.randperm(self.N, generator=generator, device=state.device)
            for index in order.tolist():
                # [中介变量] weights[index]:(N,)·state:(N,)→该神经元净局部场
                local_field = float(torch.dot(self.weights[index], state).item())
                # [判断] 正场取 +1、负场取 -1、恰好为零保持旧值
                new_value = (
                    1.0
                    if local_field > 0.0
                    else -1.0
                    if local_field < 0.0
                    else state[index].item()
                )
                if new_value != state[index].item():
                    # [更新] 立即写回；后续坐标会看到最新状态，这就是异步更新
                    state[index] = new_value
                    changed = True
                updates += 1
            energies.append(self.energy(state))
            errors.append(float(torch.mean((state != target_f).to(torch.float64)).item()))
            # [观测·固定点] 干净记忆第一整轮完全不变，才通过 U1 判据
            if sweep == 1:
                first_sweep_unchanged = not changed
            # [判断] 一整轮零翻转说明到达离散不动点，可以提前停止
            if not changed:
                return RetrievalResult(
                    state.to(torch.int8),
                    "fixed",
                    sweep,
                    updates,
                    updates * (2 * self.N - 1),
                    energies,
                    errors,
                    {"first_sweep_unchanged": first_sweep_unchanged},
                )
        return RetrievalResult(
            state.to(torch.int8),
            "max_steps",
            max_sweeps,
            updates,
            updates * (2 * self.N - 1),
            energies,
            errors,
            {"first_sweep_unchanged": first_sweep_unchanged},
        )

    def resource_summary(self) -> dict[str, int | str]:
        return {
            "model_config": "hebbian_zero_diagonal_async",
            # [观测·资源] 对称矩阵只计独立非对角权重；storage_bytes 仍按真实密集张量计
            "parameter_count": self.N * (self.N - 1) // 2,
            "storage_bytes": self.weights.numel() * self.weights.element_size(),
        }


# [模型 b/d] Polynomial DAM：记忆表存储，逐坐标最小化 power energy
class PolynomialDAM:
    def __init__(self, degree: int = 3) -> None:
        # [约束] d<2 不再是这里要比较的高阶 DAM 条件
        if degree < 2:
            raise ValueError("degree must be at least 2")
        self.degree = int(degree)
        self.model_id = f"polynomial_dam_d{self.degree}"

    def fit(self, patterns: torch.Tensor) -> "PolynomialDAM":
        binary = patterns.to(torch.float64)
        if binary.ndim != 2:
            raise ValueError("patterns must have shape [P, N]")
        # [存储] 不展开高阶权重张量，直接保存 X:(P,N)
        self.patterns = binary.detach().clone()
        self.P, self.N = map(int, binary.shape)
        return self

    def energy(self, overlaps: torch.Tensor) -> float:
        # [观测·数据] E=-Σ_mu m_mu^d；这里只比较同一模型轨迹内的差值
        return float(-torch.sum(overlaps.pow(self.degree)).item())

    def retrieve(
        self,
        cue: torch.Tensor,
        *,
        target: torch.Tensor,
        update_seed: int,
        max_sweeps: int,
    ) -> RetrievalResult:
        if max_sweeps <= 0:
            raise ValueError("max_sweeps must be positive")
        state = binary_state(cue)
        target_f = target.to(torch.float64)
        # [中介变量] overlaps:(P,)，一次计算后靠单坐标增量更新
        # [中介变量] 使用论文式原始内积；尺度写入 model_config，不和 Polynomial 混用
        overlaps = self.patterns @ state / self.N
        generator = make_generator(update_seed, state.device)
        energies = [self.energy(overlaps)]
        errors = [float(torch.mean((state != target_f).to(torch.float64)).item())]
        updates = 0
        first_sweep_unchanged = False
        for sweep in range(1, max_sweeps + 1):
            changed = False
            order = torch.randperm(self.N, generator=generator, device=state.device)
            for index in order.tolist():
                # [中介变量] coordinate:(P,) 是翻动 s_i 对所有 overlap 的贡献
                coordinate = self.patterns[:, index] / self.N
                # [中介变量] 先扣掉 s_i 旧贡献，再分别试算 +1 与 -1
                base = overlaps - coordinate * state[index]
                # [判断] 最大化 Σm^d 等价于最小化前面定义的负能量
                score_plus = torch.sum((base + coordinate).pow(self.degree))
                score_minus = torch.sum((base - coordinate).pow(self.degree))
                old_value = state[index].item()
                if score_plus > score_minus:
                    new_value = 1.0
                elif score_minus > score_plus:
                    new_value = -1.0
                else:
                    new_value = old_value
                if new_value != old_value:
                    state[index] = new_value
                    # [更新] 只更新受 s_i 改变的 overlap，避免每次重算 X@state
                    overlaps += coordinate * (new_value - old_value)
                    changed = True
                updates += 1
            energies.append(self.energy(overlaps))
            errors.append(float(torch.mean((state != target_f).to(torch.float64)).item()))
            if sweep == 1:
                first_sweep_unchanged = not changed
            diagnostics = {
                "degree": self.degree,
                "first_sweep_unchanged": first_sweep_unchanged,
            }
            if not changed:
                return RetrievalResult(
                    state.to(torch.int8),
                    "fixed",
                    sweep,
                    updates,
                    updates * (8 * self.P + 2),
                    energies,
                    errors,
                    diagnostics,
                )
        return RetrievalResult(
            state.to(torch.int8),
            "max_steps",
            max_sweeps,
            updates,
            updates * (8 * self.P + 2),
            energies,
            errors,
            diagnostics,
        )

    def resource_summary(self) -> dict[str, int | str]:
        return {
            "model_config": f"power_energy_degree_{self.degree}_async",
            "parameter_count": self.P * self.N,
            "storage_bytes": self.patterns.numel() * self.patterns.element_size(),
        }


# [模型 b/d] Exponential DAM：用指数吸引函数形成更尖锐的记忆竞争
class ExponentialDAM:
    model_id = "exponential_dam"

    def fit(self, patterns: torch.Tensor) -> "ExponentialDAM":
        self.patterns = patterns.to(torch.float64).detach().clone()
        self.P, self.N = map(int, self.patterns.shape)
        return self

    # [数值稳定] 用 -logsumexp(m) 代替直接计算 -Σexp(m)，排序与下降方向不变
    def log_energy(self, overlaps: torch.Tensor) -> float:
        return float(-torch.logsumexp(overlaps, dim=0).item())

    def retrieve(
        self, cue: torch.Tensor, *, target: torch.Tensor,
        update_seed: int, max_sweeps: int,
    ) -> RetrievalResult:
        if max_sweeps <= 0:
            raise ValueError("max_sweeps must be positive")
        state = binary_state(cue)
        target_f = target.to(torch.float64)
        overlaps = self.patterns @ state
        generator = make_generator(update_seed, state.device)
        energies = [self.log_energy(overlaps)]
        errors = [float(torch.mean((state != target_f).to(torch.float64)).item())]
        updates = 0
        first_sweep_unchanged = False
        for sweep in range(1, max_sweeps + 1):
            changed = False
            order = torch.randperm(self.N, generator=generator, device=state.device)
            for index in order.tolist():
                coordinate = self.patterns[:, index]
                base = overlaps - coordinate * state[index]
                # [判断] 分别计算 s_i=+1/-1 的 log-sum-exp，选择能量更低者
                plus = torch.logsumexp(base + coordinate, dim=0)
                minus = torch.logsumexp(base - coordinate, dim=0)
                old_value = state[index].item()
                new_value = 1.0 if plus > minus else -1.0 if minus > plus else old_value
                if new_value != old_value:
                    state[index] = new_value
                    overlaps += coordinate * (new_value - old_value)
                    changed = True
                updates += 1
            energies.append(self.log_energy(overlaps))
            errors.append(float(torch.mean((state != target_f).to(torch.float64)).item()))
            if sweep == 1:
                first_sweep_unchanged = not changed
            diagnostics = {
                "first_sweep_unchanged": first_sweep_unchanged,
                # [溯源] 明记轨迹保存的是数值稳定代理，避免读图时当成原始指数和
                "energy_note": "negative log-sum-exp; monotone proxy",
            }
            if not changed:
                return RetrievalResult(
                    state.to(torch.int8), "fixed", sweep, updates,
                    updates * (8 * self.P + 2), energies, errors, diagnostics,
                )
        return RetrievalResult(
            state.to(torch.int8), "max_steps", max_sweeps, updates,
            updates * (8 * self.P + 2), energies, errors, diagnostics,
        )

    def resource_summary(self) -> dict[str, int | str]:
        return {
            "model_config": "exp_energy_async",
            "parameter_count": self.P * self.N,
            "storage_bytes": self.patterns.numel() * self.patterns.element_size(),
        }


# [模型 b/d] R12：二元边与三元单形的混合稀释网络
class SimplicialR12:
    def __init__(
        self, triangle_fraction: float = 0.5, structure_seed: int = 31415,
        budget_type: str = "parameter",
    ) -> None:
        if not 0.0 <= triangle_fraction <= 1.0:
            raise ValueError("triangle_fraction must lie in [0, 1]")
        # [约束] H3 只允许三种预注册资源反事实，不能运行后挑预算
        if budget_type not in {"parameter", "storage", "compute"}:
            raise ValueError("unknown simplicial budget_type")
        self.triangle_fraction = float(triangle_fraction)
        self.structure_seed = int(structure_seed)
        self.budget_type = budget_type
        self.model_id = f"simplicial_r12_t{int(100 * triangle_fraction):02d}"

    def fit(self, patterns: torch.Tensor) -> "SimplicialR12":
        binary = patterns.to(torch.float64)
        self.P, self.N = map(int, binary.shape)
        # [实验控制] pairwise R12 的 C(N,2) 个连接作为三种预算基准
        pairwise_budget = self.N * (self.N - 1) // 2
        # [实验控制] matched-parameter：各阶连接权重的总个数相同
        if self.budget_type == "parameter":
            connection_count = pairwise_budget
        # [实验控制] matched-storage：同时计算 int64 顶点索引与 float64 权重
        elif self.budget_type == "storage":
            bytes_per_connection = 24.0 + 8.0 * self.triangle_fraction
            connection_count = int((24 * pairwise_budget) // bytes_per_connection)
        # [实验控制] matched-compute：用每轮需访问的顶点 incidence 数作上限
        else:
            incidences_per_connection = 2.0 + self.triangle_fraction
            connection_count = int((2 * pairwise_budget) // incidences_per_connection)
        # [中介变量] 先按比例分配，再用下面循环修正整数取整越界
        triangle_count = int(round(self.triangle_fraction * connection_count))
        edge_count = connection_count - triangle_count
        if self.budget_type == "storage":
            while 24 * edge_count + 32 * triangle_count > 24 * pairwise_budget:
                if triangle_count:
                    triangle_count -= 1
                else:
                    edge_count -= 1
        if self.budget_type == "compute":
            while 2 * edge_count + 3 * triangle_count > 2 * pairwise_budget:
                if triangle_count:
                    triangle_count -= 1
                else:
                    edge_count -= 1
        # [实验控制] structure_seed 只控制拓扑；数据种子和线索种子保持不变
        generator = make_generator(self.structure_seed)
        # [中介变量] 枚举候选边后无放回采样，避免重复连接
        all_edges = torch.combinations(torch.arange(self.N), r=2)
        edge_order = torch.randperm(len(all_edges), generator=generator)[:edge_count]
        self.edges = all_edges[edge_order]
        # [中介变量] triangles:(T,3)，每行是一条三元相互作用
        all_triangles = torch.combinations(torch.arange(self.N), r=3)
        triangle_order = torch.randperm(len(all_triangles), generator=generator)[:triangle_count]
        self.triangles = all_triangles[triangle_order]
        # [存储] w_ij=(1/N)Σ_mu ξ_i^mu ξ_j^mu，一列对应一条采样边
        self.edge_weights = (
            binary[:, self.edges[:, 0]] * binary[:, self.edges[:, 1]]
        ).sum(dim=0) / self.N
        # [存储] w_ijk=(1/N)Σ_mu ξ_i^mu ξ_j^mu ξ_k^mu
        self.triangle_weights = (
            binary[:, self.triangles[:, 0]]
            * binary[:, self.triangles[:, 1]]
            * binary[:, self.triangles[:, 2]]
        ).sum(dim=0) / self.N
        # [中介变量] 预建每个神经元的 incident 列表，检索时不再全表扫描
        self.edge_incident = []
        self.triangle_incident = []
        for neuron in range(self.N):
            # [中介变量] 找出包含当前 neuron 的全部边及其另一个端点
            edge_mask = torch.any(self.edges == neuron, dim=1)
            incident_edges = self.edges[edge_mask]
            neighbors = torch.where(
                incident_edges[:, 0] == neuron,
                incident_edges[:, 1], incident_edges[:, 0],
            )
            self.edge_incident.append((neighbors, self.edge_weights[edge_mask]))
            # [中介变量] 对三元单形保存另外两个端点，供局部场计算乘积
            triangle_mask = torch.any(self.triangles == neuron, dim=1)
            incident_triangles = self.triangles[triangle_mask]
            others = (
                torch.stack([row[row != neuron] for row in incident_triangles])
                if len(incident_triangles)
                # [失败边界] 某神经元没有三元邻居时保留合法 (0,2) 张量
                else torch.empty((0, 2), dtype=torch.long)
            )
            self.triangle_incident.append((others, self.triangle_weights[triangle_mask]))
        return self

    def energy(self, state: torch.Tensor) -> float:
        # [观测·数据] 每条边贡献 w_ij s_i s_j
        edge_term = self.edge_weights * state[self.edges].prod(dim=1)
        # [观测·数据] 每个三元单形贡献 w_ijk s_i s_j s_k
        triangle_term = self.triangle_weights * state[self.triangles].prod(dim=1)
        return float(-(edge_term.sum() + triangle_term.sum()).item())

    def retrieve(
        self, cue: torch.Tensor, *, target: torch.Tensor,
        update_seed: int, max_sweeps: int,
    ) -> RetrievalResult:
        if max_sweeps <= 0:
            raise ValueError("max_sweeps must be positive")
        state = binary_state(cue)
        target_f = target.to(torch.float64)
        generator = make_generator(update_seed)
        energies = [self.energy(state)]
        errors = [float(torch.mean((state != target_f).to(torch.float64)).item())]
        updates = 0
        first_sweep_unchanged = False
        for sweep in range(1, max_sweeps + 1):
            changed = False
            for neuron in torch.randperm(self.N, generator=generator).tolist():
                # [中介变量] 当前神经元的一阶邻接与二阶邻接已在 fit 阶段缓存
                neighbors, edge_weights = self.edge_incident[neuron]
                others, triangle_weights = self.triangle_incident[neuron]
                # [更新] 边给线性票数，三元单形给另外两点乘积后的票数
                local_field = torch.dot(edge_weights, state[neighbors])
                local_field += torch.sum(
                    triangle_weights * state[others].prod(dim=1)
                )
                old_value = state[neuron].item()
                # [判断] 论文 Θ 约定在零场取 +1；与 Classical 的零场保持不同并明确披露
                new_value = 1.0 if local_field >= 0.0 else -1.0
                if new_value != old_value:
                    state[neuron] = new_value
                    changed = True
                updates += 1
            energies.append(self.energy(state))
            errors.append(float(torch.mean((state != target_f).to(torch.float64)).item()))
            if sweep == 1:
                first_sweep_unchanged = not changed
            diagnostics = {
                "first_sweep_unchanged": first_sweep_unchanged,
                "structure_seed": self.structure_seed,
                "triangle_fraction": self.triangle_fraction,
                # [证据边界] 这里是可检查能量下降的异步扩展，不冒充论文同步实验
                "paper_native_update": False,
            }
            if not changed:
                break
        status = "fixed" if not changed else "max_steps"
        mean_incidence = (2 * len(self.edges) + 3 * len(self.triangles)) / self.N
        return RetrievalResult(
            state.to(torch.int8), status, sweep, updates,
            int(updates * (3 * mean_incidence + 1)),
            energies, errors, diagnostics,
        )

    def resource_summary(self) -> dict[str, int | str]:
        # [观测·资源] 稀疏单形除权重外还必须存顶点索引，不能只报权重数
        index_bytes = (self.edges.numel() + self.triangles.numel()) * 8
        weight_bytes = (self.edge_weights.numel() + self.triangle_weights.numel()) * 8
        return {
            "model_config": (
                f"R12_triangle_fraction_{self.triangle_fraction:.2f}_"
                f"{self.budget_type}_budget_seed_{self.structure_seed}_"
                "async_extension"
            ),
            "parameter_count": len(self.edges) + len(self.triangles),
            "storage_bytes": index_bytes + weight_bytes,
        }


# [模型 b/d] Product-of-Sums Hopfield：分组相关度相乘后一次读出
class PSHN:
    def __init__(self, groups: int = 8) -> None:
        self.groups = int(groups)
        self.model_id = f"pshn_k{self.groups}"

    def fit(self, patterns: torch.Tensor) -> "PSHN":
        self.patterns = patterns.to(torch.float64).detach().clone()
        self.P, self.N = map(int, self.patterns.shape)
        # [约束] 每组必须等长，因此 k 必须整除 N
        if self.N % self.groups:
            raise ValueError("N must be divisible by the number of PSHN groups")
        self.group_size = self.N // self.groups
        # [存储] X:(P,N)→(P,k,N/k)，只改变视图，不复制出高阶张量
        self.grouped = self.patterns.reshape(self.P, self.groups, self.group_size)
        return self

    def retrieve(
        self, cue: torch.Tensor, *, target: torch.Tensor,
        update_seed: int, max_sweeps: int,
    ) -> RetrievalResult:
        state = binary_state(cue)
        query = state.reshape(self.groups, self.group_size)
        # [中介变量] correlations:(P,k)，每条记忆在每一组上的局部相关度
        correlations = torch.einsum("kg,Pkg->Pk", query, self.grouped)
        # [中介变量] 更新第 g 组时，只乘其余 k-1 组，避免把待更新组自我代入
        products_without_group = []
        for group in range(self.groups):
            keep = torch.arange(self.groups) != group
            products_without_group.append(correlations[:, keep].prod(dim=1))
        coefficients = torch.stack(products_without_group, dim=1)
        # [更新] 记忆系数 C:(P,k) 与 X:(P,k,N/k) 汇总成各组局部场
        fields = torch.einsum("Pk,Pkg->kg", coefficients, self.grouped)
        # [判断] 一次性对所有坐标取符号；这是 one-step，不伪装成收敛轮数
        output = torch.sign(fields).reshape(self.N)
        # [边界约定] 零场保持 cue 原值，避免 torch.sign 产生不合法的 0 自旋
        ties = output == 0
        output[ties] = state[ties]
        target_f = target.to(torch.float64)
        errors = [
            float(torch.mean((state != target_f).to(torch.float64)).item()),
            float(torch.mean((output != target_f).to(torch.float64)).item()),
        ]
        return RetrievalResult(
            output.to(torch.int8), "one_step", 1, self.N,
            2 * self.P * self.N + self.P * self.groups,
            [], errors,
            {
                "first_sweep_unchanged": bool(torch.equal(output, state)),
                "groups": self.groups,
                "output_kind": "binary",
            },
        )

    def resource_summary(self) -> dict[str, int | str]:
        return {
            "model_config": f"product_of_sums_k_{self.groups}_one_step",
            "parameter_count": self.P * self.N,
            "storage_bytes": self.patterns.numel() * self.patterns.element_size(),
        }


# [模型 b/d] Continuous Modern Hopfield：softmax 注意力式一次检索
class ContinuousModernHopfield:
    model_id = "continuous_modern"

    def __init__(self, beta: float = 1.0) -> None:
        self.beta = float(beta)

    def fit(self, patterns: torch.Tensor) -> "ContinuousModernHopfield":
        self.patterns = patterns.to(torch.float64).detach().clone()
        self.P, self.N = map(int, self.patterns.shape)
        return self

    def retrieve(
        self, cue: torch.Tensor, *, target: torch.Tensor,
        update_seed: int, max_sweeps: int,
    ) -> RetrievalResult:
        # [输入] query:(N,)；连续读出允许终态落在 [-1,+1]^N 内部
        query = cue.to(torch.float64)
        # [中介变量] attention:(P,)，beta 控制记忆竞争的集中程度
        attention = torch.softmax(self.beta * (self.patterns @ query), dim=0)
        # [更新] (P,)@(P,N)→(N,)；输出是记忆的凸组合，不强制二值化
        output = attention @ self.patterns
        # [观测·数据] signed 只用于公共 bit-error 轨迹，不替换真实连续 final_state
        signed = torch.where(output >= 0, 1.0, -1.0)
        target_f = target.to(torch.float64)
        errors = [
            float(torch.mean((query != target_f).to(torch.float64)).item()),
            float(torch.mean((signed != target_f).to(torch.float64)).item()),
        ]
        # [观测·机制] 注意力熵量化读出集中度，clamp 只防 log(0)
        entropy = -torch.sum(attention * torch.log(attention.clamp_min(1e-300)))
        return RetrievalResult(
            output, "one_step", 1, self.N,
            4 * self.P * self.N, [], errors,
            {
                "first_sweep_unchanged": False,
                "fixed_point_eligible": False,
                "output_kind": "continuous",
                "attention_entropy": float(entropy.item()),
                "beta": self.beta,
            },
        )

    def resource_summary(self) -> dict[str, int | str]:
        return {
            "model_config": f"softmax_beta_{self.beta:g}_one_step",
            "parameter_count": self.P * self.N,
            "storage_bytes": self.patterns.numel() * self.patterns.element_size(),
        }


# [注册表] 主实验只通过工厂替换 b/d；a/c/e/f 代码完全复用
MODEL_FACTORIES: dict[str, Callable[[], Any]] = {
    "classical_hebb": ClassicalHopfield,
    "polynomial_dam_d3": lambda: PolynomialDAM(degree=3),
    "exponential_dam": ExponentialDAM,
    "simplicial_r12_t50": lambda: SimplicialR12(0.5, structure_seed=31415),
    "pshn_k8": lambda: PSHN(groups=8),
    "continuous_modern": lambda: ContinuousModernHopfield(beta=1.0),
}
MODEL_FACTORIES

## 3. e：统一测量与结果协议

测量函数只读取目标、终态、轨迹和资源记录，不根据模型名称偷偷换判据。共同图使用 Top-1；二值模型额外报告 exact recall。U1 的固定点判据是干净记忆经过一次登记更新后完全不变；不具备离散固定点语义的连续模型会显式标为不适用，而不是记作失败。

In [ ]:
# [观测·数据] 二值 exact recall：N 个位置必须逐元素完全一致
def e1_exact_recall(final_state: torch.Tensor, target: torch.Tensor) -> bool:
    return bool(torch.equal(final_state.to(torch.int8), target.to(torch.int8)))


# [观测·数据] overlap=(1/N)Σ_i s_i ξ_i；连续输出也有定义
def e2_overlap(final_state: torch.Tensor, target: torch.Tensor) -> float:
    return float(
        torch.mean(final_state.to(torch.float64) * target.to(torch.float64)).item()
    )


# [观测·数据] 用终态与全部记忆的内积排序，返回最相似记忆 ID
def e3_top1_memory(final_state: torch.Tensor, memories: torch.Tensor) -> int:
    # [中介变量] memories:(P,N)@state:(N,)→scores:(P,)
    scores = memories.to(torch.float64) @ final_state.to(torch.float64)
    return int(torch.argmax(scores).item())


# [观测·分类] 先区分连续/离散语义，再判目标、错误记忆、反记忆或虚假固定点
def e4_attractor_class(
    final_state: torch.Tensor,
    target_id: int,
    memories: torch.Tensor,
    status: str,
    output_kind: str,
) -> str:
    top1_id = e3_top1_memory(final_state, memories)
    # [边界] 连续终态不是离散吸引子，只报告它 Top-1 指向谁
    if output_kind == "continuous":
        return "continuous_target" if top1_id == target_id else "continuous_other"
    # [边界] 未到 fixed/one_step 的状态不能事后硬分成某个吸引子
    if status not in {"fixed", "one_step"}:
        return "nonconverged"
    final = final_state.to(torch.int8)
    binary_memories = memories.to(torch.int8)
    # [判断] 广播比较 P 条记忆，找出与终态完全相同的存储项
    matches = torch.all(binary_memories == final.unsqueeze(0), dim=1)
    matched_ids = torch.nonzero(matches, as_tuple=False).flatten().tolist()
    if target_id in matched_ids:
        return "target"
    if matched_ids:
        return "wrong_memory"
    # [判断] 反记忆单列，避免把全局翻转混入普通 spurious fixed point
    inverse_matches = torch.all(binary_memories == -final.unsqueeze(0), dim=1)
    if bool(torch.any(inverse_matches)):
        return "inverse_memory"
    return "spurious_fixed" if status == "fixed" else "one_step_nonmemory"

## 4. 配对运行器

扫描范围、重复数和停止上限都在运行前固定。异常、达到最大步数和删失不会被静默删除。`run_id + case fields` 相同的记录必须包含全部模型。

In [ ]:
# [实验控制] Gate 在运行前冻结扫描网格、重复数、停止上限与基础种子
@dataclass(frozen=True)
class BenchmarkConfig:
    N_values: tuple[int, ...]
    P_values: tuple[int, ...]
    corruption_levels: tuple[float, ...]
    pattern_sets: int
    targets_per_set: int
    max_sweeps: int
    base_seed: int
    experiment_id: str
    source_commit: str

    # [约束] 在生成任何结果前一次性拒绝空扫描、非法噪声和零重复
    def validate(self) -> None:
        if not self.N_values or not self.P_values or not self.corruption_levels:
            raise ValueError("scan dimensions must not be empty")
        if min(self.N_values) <= 0 or min(self.P_values) <= 0:
            raise ValueError("N and P must be positive")
        if not all(0.0 <= level <= 1.0 for level in self.corruption_levels):
            raise ValueError("corruption levels must lie in [0, 1]")
        if self.pattern_sets <= 0 or self.targets_per_set <= 0 or self.max_sweeps <= 0:
            raise ValueError("replicate counts and max_sweeps must be positive")


# [实验控制] 用坐标确定性派生子种子；循环顺序改变时 trial 随机性仍可追踪
def stable_seed(base: int, *coordinates: int) -> int:
    value = int(base) & 0x7FFFFFFF
    for coordinate in coordinates:
        value = (1_103_515_245 * value + 12_345 + int(coordinate)) & 0x7FFFFFFF
    return value


# [数据契约] 这些字段共同定义一条配对 trial；颜色不能掩盖输入条件变化
PAIR_KEY = [
    "run_id", "experiment_id", "source_commit", "task_id",
    "pattern_set_id", "target_id", "dataset_id", "encoding",
    "N", "P", "corruption_kind", "corruption_level",
    "data_seed", "cue_seed", "update_seed", "max_sweeps",
    "success_criterion", "retrieval_budget",
    "resource_budget_type", "stopping_rule",
]


# [判断] 作图前检查每个 trial 是否恰好包含一次完整模型集合
def validate_paired_results(
    frame: pd.DataFrame,
    expected_models: Iterable[str] | None = None,
) -> None:
    missing = set(PAIR_KEY).difference(frame.columns)
    if missing or frame.empty:
        raise ValueError(f"invalid result table; missing={sorted(missing)}")
    # [中介变量] expected 来自预注册工厂，不从成功记录反推，避免失败模型消失
    expected = set(expected_models or sorted(frame["model_id"].unique()))
    for key, group in frame.groupby(PAIR_KEY, dropna=False, sort=False):
        observed = list(group["model_id"])
        if len(observed) != len(set(observed)) or set(observed) != expected:
            raise ValueError(f"unpaired trial {key}: observed={observed}")


# [失败边界] 异常也生成标准行并留在分母；NaN 只用于本来没有定义的量
def failure_row(
    common: dict[str, Any],
    model_id: str,
    reason: str,
) -> dict[str, Any]:
    return {
        **common,
        "model_id": model_id,
        "model_config": "unavailable",
        "status": "numerical_failure",
        "sweeps": 0,
        "state_updates": 0,
        "retrieval_mode": "unavailable",
        "one_sweep_unchanged": False,
        "output_kind": "unknown",
        # [边界] 当前唯一无离散固定点语义的是 continuous_modern
        "fixed_point_eligible": model_id != "continuous_modern",
        "exact_recall": float("nan"),
        "top1_correct": False,
        "overlap": float("nan"),
        "mse": float("nan"),
        "top1_id": float("nan"),
        "attractor_class": "nonconverged",
        "parameter_count": float("nan"),
        "storage_bytes": float("nan"),
        "retrieval_flops": float("nan"),
        "wall_time_ms": float("nan"),
        "right_censored": False,
        "failure_reason": reason,
        "paper_reported": False,
        "energy_trace": [],
        "error_trace": [],
    }


# [执行] 一次生成 memory/cue，再依次交给全部模型；模型循环内禁止重采样
def run_paired_benchmark(
    model_factories: dict[str, Callable[[], Any]],
    config: BenchmarkConfig,
) -> pd.DataFrame:
    config.validate()
    # [约束] 少于两条模型线就不再是跨模型基准
    if len(model_factories) < 2:
        raise ValueError("at least two models are required")
    rows: list[dict[str, Any]] = []
    case_index = 0
    for N_index, N in enumerate(config.N_values):
        for P_index, P in enumerate(config.P_values):
            for set_index in range(config.pattern_sets):
                # [实验控制] data_seed 只由 N、P、pattern-set 坐标决定
                data_seed = stable_seed(config.base_seed, N_index, P_index, set_index)
                # [输入] 同一个 MemorySet 会被本条件下全部模型读取
                memories = a1_make_independent_binary(N, P, data_seed)
                pattern_set_id = f"N{N}-P{P}-set{set_index}-seed{data_seed}"
                # [中介变量] 每个模型每个 pattern set 只 fit 一次，多个 cue 复用存储结果
                fitted: dict[str, Any] = {}
                fit_failures: dict[str, str] = {}
                for registered_id, factory in model_factories.items():
                    try:
                        model = factory()
                        if model.model_id != registered_id:
                            raise ValueError("registry key and model_id differ")
                        fitted[registered_id] = model.fit(memories.patterns)
                    # [失败边界] fit 失败先登记原因；随后每条配对 trial 都保留失败占位行
                    except Exception as error:
                        fit_failures[registered_id] = f"{type(error).__name__}: {error}"
                for target_id in range(min(P, config.targets_per_set)):
                    target = memories.patterns[target_id]
                    for level_index, level in enumerate(config.corruption_levels):
                        # [实验控制] cue_seed 与 update_seed 分离，方便独立复现实验噪声和动力学
                        cue_seed = stable_seed(
                            config.base_seed, N_index, P_index, set_index,
                            target_id, level_index,
                        )
                        update_seed = stable_seed(cue_seed, 91)
                        # [输入] cue 在 model_id 循环外构造，保证逐元素一致
                        cue = c1_make_hamming_cue(target, level, cue_seed)
                        # [数据契约] common 字段复制进每个模型结果，供配对检查逐项核对
                        common = {
                            "run_id": f"{config.experiment_id}-{case_index:07d}",
                            "experiment_id": config.experiment_id,
                            "source_commit": config.source_commit,
                            "source_notebook": SOURCE_NOTEBOOK,
                            "task_id": (
                                "U1_clean_fixed_point"
                                if level == 0.0
                                else "U2_noise_recovery"
                            ),
                            "pattern_set_id": pattern_set_id,
                            "target_id": target_id,
                            "dataset_id": memories.dataset_id,
                            "encoding": memories.encoding,
                            "N": N,
                            "P": P,
                            "corruption_kind": "hamming_flip",
                            "corruption_level": float(level),
                            "data_seed": data_seed,
                            "structure_seed": None,
                            "cue_seed": cue_seed,
                            "update_seed": update_seed,
                            "max_sweeps": config.max_sweeps,
                            "success_criterion": "top1_memory_identification",
                            "retrieval_budget": "native_registered_rule",
                            "resource_budget_type": "native",
                            "stopping_rule": "model_native",
                        }
                        case_index += 1
                        for model_id in model_factories:
                            if model_id in fit_failures:
                                rows.append(failure_row(common, model_id, fit_failures[model_id]))
                                continue
                            model = fitted[model_id]
                            try:
                                # [观测·资源] wall-clock 只做描述；公平图主要使用登记 FLOPs
                                started = perf_counter()
                                result = model.retrieve(
                                    cue,
                                    target=target,
                                    update_seed=update_seed,
                                    max_sweeps=config.max_sweeps,
                                )
                                elapsed_ms = (perf_counter() - started) * 1_000.0
                                final_f = result.final_state.to(torch.float64)
                                target_f = target.to(torch.float64)
                                # [边界] 输出类型来自适配器，不根据 model_id 在指标函数里偷换
                                output_kind = result.diagnostics.get("output_kind", "binary")
                                fixed_point_eligible = bool(
                                    result.diagnostics.get(
                                        "fixed_point_eligible", output_kind == "binary"
                                    )
                                )
                                # [观测·数据] 所有模型共享的主成功判据：Top-1 是否等于 target_id
                                top1_id = e3_top1_memory(
                                    result.final_state, memories.patterns
                                )
                                rows.append({
                                    **common,
                                    "model_id": model_id,
                                    **model.resource_summary(),
                                    "status": result.status,
                                    "sweeps": result.sweeps,
                                    "state_updates": result.state_updates,
                                    "retrieval_mode": (
                                        "one_step"
                                        if result.status == "one_step"
                                        else "iterative_async"
                                    ),
                                    "output_kind": output_kind,
                                    "fixed_point_eligible": fixed_point_eligible,
                                    "one_sweep_unchanged": bool(
                                        result.diagnostics.get("first_sweep_unchanged", False)
                                    ),
                                    "exact_recall": (
                                        e1_exact_recall(result.final_state, target)
                                        if output_kind == "binary"
                                        else float("nan")
                                    ),
                                    "top1_correct": top1_id == target_id,
                                    "overlap": e2_overlap(result.final_state, target),
                                    "mse": float((final_f - target_f).pow(2).mean().item()),
                                    "top1_id": top1_id,
                                    "attractor_class": e4_attractor_class(
                                        result.final_state, target_id, memories.patterns,
                                        result.status, output_kind,
                                    ),
                                    "retrieval_flops": result.retrieval_flops,
                                    "wall_time_ms": elapsed_ms,
                                    "right_censored": False,
                                    "failure_reason": "",
                                    "paper_reported": False,
                                    "energy_trace": result.energy_trace,
                                    "error_trace": result.error_trace,
                                })
                            # [失败边界] 单次检索失败不终止整批，也不从统计分母删除
                            except Exception as error:
                                reason = f"{type(error).__name__}: {error}"
                                rows.append(failure_row(common, model_id, reason))
    frame = pd.DataFrame(rows)
    # [判断] 只有完整配对表才能离开运行器进入 f 作图组件
    validate_paired_results(frame, tuple(model_factories))
    return frame


# [汇总] 从 clean cue 的一步不变率定义有限扫描经验容量，不做渐近外推
def capacity_summary(frame: pd.DataFrame, threshold: float = 0.9) -> pd.DataFrame:
    if not 0.0 < threshold < 1.0:
        raise ValueError("threshold must lie in (0, 1)")
    # [边界] 连续模型不具备这里的离散固定点语义，因此显式排除
    clean = frame[
        (frame["corruption_level"] == 0.0)
        & frame["fixed_point_eligible"]
    ]
    rates = (
        clean.groupby(["model_id", "N", "P"], as_index=False)["one_sweep_unchanged"]
        .mean()
        .rename(columns={"one_sweep_unchanged": "success_rate"})
    )
    rows = []
    for (model_id, N), group in rates.groupby(["model_id", "N"], sort=False):
        ordered = group.sort_values("P")
        # [判断] P_c 是扫描网格中最后一个达到预注册阈值的点
        passing = ordered[ordered["success_rate"] >= threshold]
        # [删失] 最小 P 都失败只得到左删失上界，不伪造临界点
        if passing.empty:
            critical = int(ordered["P"].min())
            left_censored, right_censored = True, False
        else:
            critical = int(passing["P"].max())
            left_censored = False
            # [删失] 最大 P 仍成功只得到下界，不能拿去拟合容量阶数
            right_censored = critical == int(ordered["P"].max())
        rows.append({
            "model_id": model_id, "N": int(N), "P_c": critical,
            "success_threshold": threshold,
            "left_censored": left_censored,
            "right_censored": right_censored,
            "capacity_kind": "clean_fixed_point_capacity",
        })
    return pd.DataFrame(rows)

## 5. f：同图比较组件

颜色只编码模型。共同曲线默认读取 `top1_correct`，因此连续输出不会被强制二值化。容量图只纳入 fixed-point-eligible 记录并使用实际 `P`；失败保留在分母，扫描边界用删失符号表示。

In [ ]:
# [展示] 颜色只绑定 model_id，预算与动力学差异写进标题/表格
MODEL_LABELS = {
    "classical_hebb": "Classical Hopfield",
    "polynomial_dam_d3": "Polynomial DAM (d=3)",
    "exponential_dam": "Exponential DAM",
    "simplicial_r12_t50": "Simplicial R12 (50% triangles)",
    "pshn_k8": "PSHN (k=8)",
    "continuous_modern": "Continuous Modern Hopfield",
}


# [汇总] 先在 pattern set 内平均 target，再把 pattern set 当独立重复计算区间
def curve_table(
    frame: pd.DataFrame,
    x: str,
    fixed: dict[str, Any],
    metric: str = "top1_correct",
) -> pd.DataFrame:
    subset = frame.copy()
    for column, value in fixed.items():
        subset = subset[subset[column] == value]
    # [判断] 聚合之前再次验证配对，防止筛选条件误删某条模型线
    validate_paired_results(subset)
    # [中介变量] replicate_rates 每行对应一个独立 pattern-set 重复
    replicate_rates = (
        subset.groupby(["model_id", x, "pattern_set_id"], as_index=False)[metric]
        .mean()
    )
    curve = (
        replicate_rates.groupby(["model_id", x])[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    curve["metric"] = metric
    # [观测·不确定性] SE 来自 pattern-set 间波动，不把同一记忆库的多个 target 当独立样本
    curve["se"] = curve["std"].fillna(0.0) / np.sqrt(curve["count"])
    curve["lower"] = np.clip(curve["mean"] - 1.96 * curve["se"], 0, 1)
    curve["upper"] = np.clip(curve["mean"] + 1.96 * curve["se"], 0, 1)
    return curve


# [展示] f1/f2 共用同一绘图函数；它不重新调用模型，也不改成功判据
def plot_success_curve(
    curve: pd.DataFrame,
    x: str,
    xlabel: str,
    title: str,
    *,
    log_x: bool = False,
) -> None:
    fig, axis = plt.subplots(figsize=(8.2, 4.8))
    for model_id, group in curve.groupby("model_id", sort=False):
        ordered = group.sort_values(x)
        axis.plot(ordered[x], ordered["mean"], marker="o", label=MODEL_LABELS[model_id])
        axis.fill_between(ordered[x], ordered["lower"], ordered["upper"], alpha=0.14)
    metric = curve["metric"].iloc[0]
    ylabel = "Top-1 memory identification rate" if metric == "top1_correct" else metric
    axis.set(xlabel=xlabel, ylabel=ylabel, ylim=(-0.03, 1.03))
    if log_x:
        axis.set_xscale("log", base=2)
    axis.set_title(title)
    axis.legend(fontsize=8, ncol=2)
    fig.tight_layout()


# [展示] 空心三角形编码删失方向，提醒读者扫描边界不是实测临界点
def plot_capacity(summary: pd.DataFrame) -> None:
    fig, axis = plt.subplots(figsize=(8.2, 4.8))
    for model_id, group in summary.groupby("model_id", sort=False):
        ordered = group.sort_values("N")
        axis.plot(ordered["N"], ordered["P_c"], marker="o", label=MODEL_LABELS[model_id])
        for marker, column in [("^", "right_censored"), ("v", "left_censored")]:
            censored = ordered[ordered[column]]
            axis.scatter(
                censored["N"], censored["P_c"], marker=marker, s=90,
                facecolors="none", edgecolors="black",
            )
    axis.set(xlabel="State dimension N", ylabel="Empirical critical capacity P_c")
    axis.set_yscale("log", base=2)
    axis.set_title("U1/U3 clean fixed-point capacity | unchanged ≥ 90%")
    axis.legend(fontsize=8, ncol=2)
    fig.tight_layout()


# [展示] 不同模型能量不可比，因此这里只叠加统一 signed bit-error 轨迹
def plot_dynamics(frame: pd.DataFrame, run_id: str) -> None:
    subset = frame[frame["run_id"] == run_id]
    validate_paired_results(subset)
    fig, axis = plt.subplots(figsize=(8.2, 4.8))
    for row in subset.itertuples(index=False):
        axis.plot(
            range(len(row.error_trace)), row.error_trace, marker="o",
            label=MODEL_LABELS[row.model_id],
        )
    axis.set(xlabel="Registered retrieval step", ylabel="Signed bit error ratio", ylim=(-0.03, 1.03))
    axis.set_title(f"U5 paired dynamics | {run_id}")
    axis.legend(fontsize=8, ncol=2)
    fig.tight_layout()


# [展示] U7 把质量和登记 FLOPs 同时画出，避免只看召回率排冠军
def plot_quality_cost(frame: pd.DataFrame, N: int, P: int, level: float) -> None:
    subset = frame[
        (frame["N"] == N)
        & (frame["P"] == P)
        & (frame["corruption_level"] == level)
        & frame["retrieval_flops"].notna()
    ]
    validate_paired_results(subset)
    grouped = subset.groupby("model_id", as_index=False).agg(
        top1_correct=("top1_correct", "mean"),
        retrieval_flops=("retrieval_flops", "mean"),
    )
    fig, axis = plt.subplots(figsize=(8.2, 4.8))
    for row in grouped.itertuples(index=False):
        axis.scatter(row.retrieval_flops, row.top1_correct, s=80)
        axis.annotate(
            MODEL_LABELS[row.model_id], (row.retrieval_flops, row.top1_correct),
            xytext=(5, 5), textcoords="offset points", fontsize=8,
        )
    axis.set(
        xlabel="Approximate retrieval FLOPs", ylabel="Top-1 identification rate",
        ylim=(-0.03, 1.03),
    )
    axis.set_xscale("log")
    axis.set_title(f"U7 native quality-cost | N={N}, P={P}, corruption={level:.2f}")
    fig.tight_layout()

## 6. 公共组件自检

这不是实验结论。它只验证六个适配器能接收同一个 memory/cue、返回统一字段；若模型提供离散能量轨迹，再检查该轨迹不增加。Continuous Modern 和 PSHN 不伪造能量曲线。

In [ ]:
# [自检输入] 小尺寸 memory/cue 只检查组件契约，不进入任何实验结论
check_memories = a1_make_independent_binary(N=32, P=4, data_seed=7)
check_target = check_memories.patterns[0]
check_cue = c1_make_hamming_cue(check_target, 0.125, cue_seed=11)
for registered_id, factory in MODEL_FACTORIES.items():
    check_model = factory().fit(check_memories.patterns)
    assert check_model.model_id == registered_id
    check_result = check_model.retrieve(
        check_cue, target=check_target, update_seed=13, max_sweeps=8
    )
    # [判断] 只有模型真实提供两点以上能量轨迹时才检查单调性
    if len(check_result.energy_trace) > 1:
        deltas = torch.diff(torch.tensor(check_result.energy_trace))
        assert bool(torch.all(deltas <= 1e-10))
        energy_check = "monotone"
    else:
        energy_check = "not supplied"
    print(registered_id, check_result.status, check_result.sweeps, energy_check)
print("component contract: passed")

## 7. 有限扫描配置

扫描范围、重复数与停止上限预先固定。它足以验证统一接口和有限规模趋势，不足以估计渐近容量阶数。主图是 `native / equal_max_sweeps`；U7 显式展示资源差异，H1–H3 另做同参数的高阶结构消融。

In [ ]:
# [溯源] 每次 Colab Run All 生成唯一 execution_id，避免覆盖后无法区分批次
EXECUTION_ID = (
    datetime.now(timezone.utc).strftime("benchmark-v1-%Y%m%dT%H%M%SZ-")
    + uuid4().hex[:8]
)
# [实验控制] 有限扫描 Gate；这里改参数会直接改变结论适用范围
CONFIG = BenchmarkConfig(
    N_values=(64, 128),
    P_values=(4, 8, 16, 32),
    corruption_levels=(0.0, 0.1, 0.2, 0.3),
    pattern_sets=3,
    targets_per_set=4,
    max_sweeps=20,
    base_seed=20260905,
    experiment_id=EXECUTION_ID,
    source_commit=SOURCE_COMMIT,
)
display(pd.Series(asdict(CONFIG), name="value").to_frame())

## 8. 运行并保存原始记录

每一行是一条目标记忆的一次检索。原始输出保存为 JSON Lines，列表轨迹不会被 CSV 静默改形。失败记录仍保留在比较分母中。

In [ ]:
# [执行] 这是主数值入口；本仓库提交的 notebook 保持未执行、无缓存输出
results = run_paired_benchmark(MODEL_FACTORIES, CONFIG)
validate_paired_results(results, tuple(MODEL_FACTORIES))

artifact_root = Path("/content/hopfield-benchmark-results")
artifact_root.mkdir(parents=True, exist_ok=True)
# [存储] JSON Lines 保留每条 trial 和列表轨迹，避免 CSV 把列表静默字符串化
results.to_json(
    artifact_root / "benchmark_v1_raw_results.jsonl",
    orient="records", lines=True,
)
with (artifact_root / "benchmark_v1_config.json").open("w", encoding="utf-8") as handle:
    json.dump(asdict(CONFIG), handle, ensure_ascii=False, indent=2)
print(f"saved {len(results)} rows to {artifact_root}")
display(results.drop(columns=["energy_trace", "error_trace"]).head(8))
display(results.groupby(["model_id", "status"]).size().rename("count").to_frame())

## 实验 1：U2 噪声恢复

固定 `N=128, P=16`。横轴是逐元素相同的 Hamming 损坏线索，纵轴是适用于二值与连续输出的 Top-1 记忆识别率。

In [ ]:
# [汇总] 固定 N/P 后只让 corruption_level 变化，形成 U2 配对噪声曲线
noise_curve = curve_table(results, "corruption_level", {"N": 128, "P": 16})
plot_success_curve(
    noise_curve, "corruption_level", "Hamming corruption ratio",
    "U2 noise recovery | N=128, P=16 | native budget",
)
plt.show()

In [ ]:
endpoint = noise_curve[noise_curve["corruption_level"] == 0.3]
# [观测·数据] Top-1 回答认出哪条记忆，overlap 补充终态离目标还有多远
endpoint_overlap = (
    results[
        (results["N"] == 128)
        & (results["P"] == 16)
        & (results["corruption_level"] == 0.3)
    ]
    .groupby("model_id")["overlap"].mean()
)
lines = [
    f"- `{row['model_id']}`：rho=0.30 时 Top-1={row['mean']:.3f}，"
    f"final overlap={endpoint_overlap.loc[row['model_id']]:.3f}，"
    f"pattern-set replicates={int(row['count'])}。"
    for _, row in endpoint.iterrows()
]
display(Markdown(
    "**本次运行的描述性结论**\n\n" + "\n".join(lines)
    + "\n\n**证据边界**：这里只是有限规模、native budget 的配对结果；连续输出没有被强制二值化，不能据此宣称容量阶数或统计优势。"
))

## 实验 2：U3 有限负载曲线

固定 `N=128` 和 10% Hamming 损坏。横轴使用实际存储数 `P`，不把线性、多项式和指数容量强行归一成同一个 `P/N`。

In [ ]:
# [汇总] 固定 N/rho 后扫描实际 P；不同容量阶的模型不强行共用 P/N
load_curve = curve_table(results, "P", {"N": 128, "corruption_level": 0.1})
plot_success_curve(
    load_curve, "P", "Stored patterns P",
    "U3 finite-load curve | N=128, corruption=0.10 | native budget",
    log_x=True,
)
plt.show()

In [ ]:
display(Markdown(
    "**本次运行的描述性结论**：下表是上图对应的条件均值。\n\n"
    "**证据边界**：曲线可能非单调；不在预设扫描点之间插值，也不把最后一个成功点自动称为理论容量。"
))
display(load_curve.pivot(index="model_id", columns="P", values="mean").round(3))

## 实验 3：U1/U3 干净固定点容量

采用预先固定的 90% one-update-unchanged 判据，只纳入具备该语义的模型。向上空心点表示容量至少达到扫描上界；向下空心点表示容量低于或等于扫描下界。

In [ ]:
# [汇总] 阈值在看结果前固定为 0.9，避免事后移动门槛
capacities = capacity_summary(results, threshold=0.9)
capacities.to_csv(artifact_root / "benchmark_v1_capacity_summary.csv", index=False)
display(capacities)
plot_capacity(capacities)
plt.show()

In [ ]:
lines = []
for row in capacities.itertuples(index=False):
    relation = "≤" if row.left_censored else "≥" if row.right_censored else "="
    lines.append(f"- `{row.model_id}`，N={row.N}：P_c {relation} {row.P_c}。")
display(Markdown(
    "**本次运行的描述性结论**\n\n" + "\n".join(lines)
    + "\n\n**证据边界**：这是离散扫描上的经验阈值。删失点不能用于普通容量拟合，也不能支持线性、多项式或指数容量结论。"
))

## 实验 4：U5 配对动力学

同一个 `run_id` 下叠加公共 bit-error 指标。两种模型的能量定义不同，所以不把能量数值混在一个纵轴。

In [ ]:
# [实验控制] 先选定一条完整配对 trial，再叠加六个模型的轨迹
dynamics_run_id = results[
    (results["N"] == 128)
    & (results["P"] == 16)
    & (results["corruption_level"] == 0.2)
]["run_id"].iloc[0]
plot_dynamics(results, dynamics_run_id)
plt.show()

In [ ]:
lines = []
for row in results[results["run_id"] == dynamics_run_id].itertuples(index=False):
    final_error = row.error_trace[-1] if row.error_trace else float("nan")
    lines.append(
        f"- `{row.model_id}`：status={row.status}，sweeps={row.sweeps}，final bit error={final_error:.3f}。"
    )
display(Markdown(
    "**本次运行的描述性结论**\n\n" + "\n".join(lines)
    + "\n\n**证据边界**：单条轨迹只解释更新过程，不代表总体召回率。"
))

## 实验 5：U7 质量—计算量

横轴是实现登记的近似操作数，不是硬件实测延迟。此图揭示 native 配置的资源差异；H1–H3 才是明确的同参数结构比较。

In [ ]:
plot_quality_cost(results, N=128, P=16, level=0.1)
plt.show()

In [ ]:
# [观测·资源] 同时列参数数、真实存储字节和检索 FLOPs，wall time 不作硬件结论
resource_table = (
    results[
        (results["N"] == 128)
        & (results["P"] == 16)
        & (results["corruption_level"] == 0.1)
    ]
    .groupby("model_id", as_index=False)
    .agg(
        top1_correct=("top1_correct", "mean"),
        retrieval_flops=("retrieval_flops", "mean"),
        storage_bytes=("storage_bytes", "mean"),
        parameter_count=("parameter_count", "mean"),
    )
)
display(Markdown(
    "**本次运行的描述性结论**：同图中的质量差异必须和下面的参数、存储与计算量一起读。\n\n"
    "**证据边界**：当前不做硬件速度结论；向量化程度会改变 wall-clock time。"
))
display(resource_table.round(3))

## 实验 6：U4 吸引域

U4 复用 U2 的完整噪声曲线，但汇总量不同：AUC 是扫描区间内曲线下面积，`rho@90%` 是仍保持至少 90% Top-1 成功率的最大已扫描噪声。没有跨扫描点外推。

In [ ]:
# [汇总] U4 不重复运行模型，直接从完整 U2 曲线计算 AUC 与 rho@90%
def basin_summary(curve: pd.DataFrame, threshold: float = 0.9) -> pd.DataFrame:
    rows = []
    for model_id, group in curve.groupby("model_id", sort=False):
        ordered = group.sort_values("corruption_level")
        x = ordered["corruption_level"].to_numpy(dtype=float)
        y = ordered["mean"].to_numpy(dtype=float)
        # [判断] 只取已扫描且成功率≥阈值的噪声点，不在网格之间插值
        passing = x[y >= threshold]
        rows.append({
            "model_id": model_id,
            # [观测·数据] AUC 只覆盖当前 corruption 网格的横轴区间
            "auc": float(np.trapz(y, x)),
            "rho_at_90": float(passing.max()) if passing.size else float("nan"),
            "right_censored": bool(passing.size and passing.max() == x.max()),
        })
    return pd.DataFrame(rows)


basins = basin_summary(noise_curve)
display(basins.round(3))
display(Markdown(
    "**本次运行的描述性结论**\n\n"
    + "\n".join(
        f"- `{row.model_id}`：AUC={row.auc:.3f}，rho@90%={row.rho_at_90:.2f}"
        + ("（达到扫描上界）" if row.right_censored else "") + "。"
        for row in basins.itertuples(index=False)
    )
    + "\n\n**证据边界**：AUC 只覆盖当前 rho 网格；rho@90% 是经验扫描值，不是理论吸引域半径。"
))

## 实验 7：U6 随机初态与虚假吸引子

虚假吸引子需要“真的迭代到固定点”才有含义，因此这里只比较 Classical、Polynomial DAM、Exponential DAM 与异步 Simplicial R12。PSHN 的一次映射和 Continuous Modern 的连续读出不被伪装成吸引子动力学。

In [ ]:
# [适用边界] U6 只纳入真正迭代到 fixed 的离散模型
ATTRACTOR_FACTORIES = {
    key: MODEL_FACTORIES[key]
    for key in [
        "classical_hebb", "polynomial_dam_d3",
        "exponential_dam", "simplicial_r12_t50",
    ]
}


# [观测·分类] 随机初态终点分为 stored、inverse、spurious 与 nonconverged
def classify_random_attractor(
    state: torch.Tensor, memories: torch.Tensor, status: str
) -> str:
    # [判断] 没到固定点时不根据暂态外观猜测吸引子类别
    if status != "fixed":
        return "nonconverged"
    final = state.to(torch.int8)
    stored = memories.to(torch.int8)
    if bool(torch.any(torch.all(stored == final.unsqueeze(0), dim=1))):
        return "stored_memory"
    if bool(torch.any(torch.all(stored == -final.unsqueeze(0), dim=1))):
        return "inverse_memory"
    return "spurious_fixed"


# [执行] 每个随机初态在模型循环外生成，再交给全部离散适配器
def run_random_start_benchmark(
    factories: dict[str, Callable[[], Any]],
    *, N: int, P: int, pattern_sets: int, starts_per_set: int,
    max_sweeps: int, base_seed: int,
) -> pd.DataFrame:
    rows = []
    for set_index in range(pattern_sets):
        data_seed = stable_seed(base_seed, set_index)
        memories = a1_make_independent_binary(N, P, data_seed)
        # [存储] 同一 pattern set 下复用已拟合模型，随机 start 只改变 cue
        fitted = {key: factory().fit(memories.patterns) for key, factory in factories.items()}
        for start_index in range(starts_per_set):
            cue_seed = stable_seed(base_seed, set_index, start_index, 17)
            update_seed = stable_seed(cue_seed, 91)
            # [输入] cue 与存储记忆独立；cue_seed 对所有模型完全相同
            cue = c2_make_random_state(N, cue_seed)
            for model_id, model in fitted.items():
                result = model.retrieve(
                    cue, target=memories.patterns[0],
                    update_seed=update_seed, max_sweeps=max_sweeps,
                )
                rows.append({
                    "model_id": model_id,
                    "pattern_set_id": set_index,
                    "start_id": start_index,
                    "data_seed": data_seed,
                    "cue_seed": cue_seed,
                    "update_seed": update_seed,
                    "outcome": classify_random_attractor(
                        result.final_state, memories.patterns, result.status
                    ),
                })
    return pd.DataFrame(rows)


spurious_results = run_random_start_benchmark(
    ATTRACTOR_FACTORIES,
    N=128, P=16, pattern_sets=3, starts_per_set=24,
    max_sweeps=30, base_seed=20260906,
)
spurious_results.to_json(
    artifact_root / "u6_random_start_raw.jsonl", orient="records", lines=True
)
# [汇总] 失败/未收敛仍在总数中，四类比例按 model_id 各自归一到 1
spurious_rates = (
    spurious_results.groupby(["model_id", "outcome"], as_index=False)
    .size().rename(columns={"size": "count"})
)
spurious_rates["rate"] = (
    spurious_rates["count"]
    / spurious_rates.groupby("model_id")["count"].transform("sum")
)
pivot = spurious_rates.pivot(index="model_id", columns="outcome", values="rate").fillna(0)
order = ["stored_memory", "inverse_memory", "spurious_fixed", "nonconverged"]
pivot = pivot.reindex(columns=order, fill_value=0)
pivot.plot(kind="bar", stacked=True, figsize=(8.2, 4.8), ylim=(0, 1))
plt.ylabel("Outcome proportion")
plt.title("U6 random-start attractor outcomes | N=128, P=16")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()
display(Markdown(
    "**本次运行的描述性结论**：随机初态的完整分类见下表；stored、inverse、spurious 与未收敛之和为 1。\n\n"
    "**证据边界**：这测的是当前有限 N、P 和停止上限下的经验吸引子分布，不等于热力学自旋玻璃相比例。"
))
display(pivot.round(3))

## 实验 8：H1/H2/H3 Simplicial 结构消融

仅改变边与三元单形的配比，并用多个 `structure_seed` 暴露随机拓扑方差。三个分面分别匹配 sparse pairwise R12 基线的连接权重数、显式索引+权重字节、以及每轮连接 incidence；compute 分面固定只做一轮更新，因此实际登记 FLOPs 也受同一上限约束。这里采用逐点异步下降扩展，**不是论文使用的同步更新复刻**。

In [ ]:
# [执行] H1/H2/H3 同时交叉阶数配比、拓扑种子和资源预算
def run_simplicial_ablation(
    *, N: int, P: int, corruption_level: float,
    fractions: tuple[float, ...], structure_seeds: tuple[int, ...],
    budget_types: tuple[str, ...], pattern_sets: int,
    targets_per_set: int, max_sweeps: int, base_seed: int,
) -> pd.DataFrame:
    rows = []
    for set_index in range(pattern_sets):
        data_seed = stable_seed(base_seed, set_index)
        # [输入] 一个 set 内所有 fraction/seed/budget 共享同一批模式
        memories = a1_make_independent_binary(N, P, data_seed)
        for structure_seed in structure_seeds:
            # [存储] structure_seed 只改变连接位置，不改变模式或 cue
            models = {
                (budget_type, fraction): SimplicialR12(
                    fraction, structure_seed, budget_type
                ).fit(memories.patterns)
                for budget_type in budget_types
                for fraction in fractions
            }
            for target_id in range(min(P, targets_per_set)):
                target = memories.patterns[target_id]
                cue_seed = stable_seed(base_seed, set_index, target_id, 37)
                # [输入] cue 位于所有结构条件循环外，保证结构消融严格配对
                cue = c1_make_hamming_cue(target, corruption_level, cue_seed)
                update_seed = stable_seed(cue_seed, 91)
                for (budget_type, fraction), model in models.items():
                    # [实验控制] matched-compute 固定一轮；另外两种预算允许按固定点停止
                    sweep_limit = 1 if budget_type == "compute" else max_sweeps
                    result = model.retrieve(
                        cue, target=target, update_seed=update_seed,
                        max_sweeps=sweep_limit,
                    )
                    resources = model.resource_summary()
                    # [观测·资源] 2E+3T 是完整一轮访问的单形顶点次数
                    incidences = 2 * len(model.edges) + 3 * len(model.triangles)
                    rows.append({
                        "budget_type": budget_type,
                        "triangle_fraction": fraction,
                        "structure_seed": structure_seed,
                        "pattern_set_id": set_index,
                        "target_id": target_id,
                        "top1_correct": e3_top1_memory(
                            result.final_state, memories.patterns
                        ) == target_id,
                        "parameter_count": resources["parameter_count"],
                        "storage_bytes": resources["storage_bytes"],
                        "incidences_per_sweep": incidences,
                        "retrieval_flops": result.retrieval_flops,
                    })
    return pd.DataFrame(rows)


simplicial_ablation = run_simplicial_ablation(
    N=64, P=16, corruption_level=0.2,
    fractions=(0.0, 0.25, 0.5, 0.75, 1.0),
    structure_seeds=(101, 202, 303, 404, 505),
    budget_types=("parameter", "storage", "compute"),
    pattern_sets=3, targets_per_set=4, max_sweeps=20,
    base_seed=20260907,
)
simplicial_ablation.to_json(
    artifact_root / "h123_simplicial_raw.jsonl", orient="records", lines=True
)
# [汇总] 先在每个结构种子内平均，再用种子间标准差显示拓扑敏感性
seed_rates = (
    simplicial_ablation.groupby(
        ["budget_type", "triangle_fraction", "structure_seed"], as_index=False
    )["top1_correct"].mean()
)
summary = (
    seed_rates.groupby(["budget_type", "triangle_fraction"])["top1_correct"]
    .agg(["mean", "std"]).reset_index()
)
fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.2), sharey=True)
for axis, budget_type in zip(axes, ("parameter", "storage", "compute")):
    panel = summary[summary["budget_type"] == budget_type]
    axis.errorbar(
        panel["triangle_fraction"], panel["mean"], yerr=panel["std"],
        marker="o", capsize=4,
    )
    axis.set(
        xlabel="Triangle fraction", title=f"matched {budget_type}",
        ylim=(-0.03, 1.03),
    )
axes[0].set_ylabel("Top-1 identification rate")
fig.suptitle("H1/H2/H3 Simplicial ablation")
fig.tight_layout()
plt.show()

# [判断] 图后公开三个预算的实际资源数，并用断言阻止整数取整越界
budgets = simplicial_ablation.groupby(
    ["budget_type", "triangle_fraction"], as_index=False
).agg(
    parameter_count=("parameter_count", "first"),
    storage_bytes=("storage_bytes", "first"),
    incidences_per_sweep=("incidences_per_sweep", "first"),
    retrieval_flops=("retrieval_flops", "first"),
)
parameter_panel = budgets[budgets["budget_type"] == "parameter"]
storage_panel = budgets[budgets["budget_type"] == "storage"]
compute_panel = budgets[budgets["budget_type"] == "compute"]
# [约束] 三条断言分别验证 matched-parameter/storage/compute 的承诺
assert parameter_panel["parameter_count"].nunique() == 1
assert storage_panel["storage_bytes"].max() <= storage_panel.iloc[0]["storage_bytes"]
assert compute_panel["incidences_per_sweep"].max() <= compute_panel.iloc[0]["incidences_per_sweep"]
assert compute_panel["retrieval_flops"].max() <= compute_panel.iloc[0]["retrieval_flops"]
display(Markdown(
    "**本次运行的描述性结论**：每个分面的均值比较阶数配比，误差条是随机拓扑种子的标准差；预算断言已通过。\n\n"
    "**证据边界**：匹配方式改变可用连接数；三个分面回答不同反事实，不能把最佳点跨分面拼成一个模型。"
))
display(summary.round(3))
display(budgets)

## 实验 9：H7 结构 × 相似度的完整 2×2

这是机制实验，不进入主模型排名。因素 A 是 pairwise 与 mixed simplicial 结构；因素 B 是 dot 与 cosine 相似度。四格共享相同模式、线索、单形数量和温度；在概率尺度上报告交互项 `p11 - p10 - p01 + p00`。这是受 Simplicial Hopfield 论文启发的协议扩展，不冒充论文原图。

In [ ]:
# [机制模型] H7 专用连续读出，只服务 2×2 结构×相似度反事实
class ContinuousSimplicialAttention:
    def __init__(
        self, triangle_fraction: float, similarity: str,
        structure_seed: int, beta: float = 1.0,
    ) -> None:
        self.triangle_fraction = float(triangle_fraction)
        self.similarity = similarity
        self.structure_seed = int(structure_seed)
        self.beta = float(beta)

    def fit(self, patterns: torch.Tensor) -> "ContinuousSimplicialAttention":
        self.patterns = patterns.to(torch.float64)
        self.P, self.N = map(int, self.patterns.shape)
        # [实验控制] pairwise 与 mixed 都使用同样数量的单形项
        budget = self.N * (self.N - 1) // 2
        triangle_count = int(round(self.triangle_fraction * budget))
        edge_count = budget - triangle_count
        generator = make_generator(self.structure_seed)
        # [中介变量] mixed 条件随机抽边/三角形；pairwise 条件使用完整边集
        all_edges = torch.combinations(torch.arange(self.N), r=2)
        self.edges = all_edges[torch.randperm(len(all_edges), generator=generator)[:edge_count]]
        all_triangles = torch.combinations(torch.arange(self.N), r=3)
        self.triangles = all_triangles[
            torch.randperm(len(all_triangles), generator=generator)[:triangle_count]
        ]
        return self

    # [中介变量] 为每条记忆累计所有已选单形上的局部相似度
    def _simplex_scores(self, cue: torch.Tensor, simplices: torch.Tensor) -> torch.Tensor:
        if len(simplices) == 0:
            return torch.zeros(self.P, dtype=torch.float64)
        # [张量] memory_parts:(P,S,order)，cue_parts:(S,order)
        memory_parts = self.patterns[:, simplices]
        cue_parts = cue.to(torch.float64)[simplices]
        dot = (memory_parts * cue_parts.unsqueeze(0)).sum(dim=2)
        # [实验控制] dot 先除单形维数，排除三元项仅因维度更大而得分更高
        if self.similarity == "dot":
            return (dot / simplices.shape[1]).sum(dim=1)
        # [数值稳定] cosine 分母下限防止零范数；二值输入通常不会触发
        if self.similarity == "cosine":
            denominator = (
                memory_parts.norm(dim=2) * cue_parts.norm(dim=1).unsqueeze(0)
            ).clamp_min(1e-12)
            return (dot / denominator).sum(dim=1)
        raise ValueError("similarity must be dot or cosine")

    def retrieve(self, cue: torch.Tensor) -> torch.Tensor:
        edge_scores = self._simplex_scores(cue, self.edges)
        triangle_scores = self._simplex_scores(cue, self.triangles)
        simplex_count = len(self.edges) + len(self.triangles)
        # [实验控制] 再除总单形数，四格共享同一总分尺度
        scores = (edge_scores + triangle_scores) / simplex_count
        return torch.softmax(self.beta * scores, dim=0) @ self.patterns


# [执行] 四格设计：pairwise/mixed × dot/cosine，其他变量全部共享
def run_factorial_h7(base_seed: int = 20260908) -> pd.DataFrame:
    rows = []
    # [实验控制] 四个组合一次性预注册，禁止只运行看起来最好的两格
    factors = [(0.0, "dot"), (0.0, "cosine"), (0.5, "dot"), (0.5, "cosine")]
    for set_index in range(4):
        data_seed = stable_seed(base_seed, set_index)
        memories = a1_make_independent_binary(64, 16, data_seed)
        models = {
            factor: ContinuousSimplicialAttention(
                factor[0], factor[1], structure_seed=stable_seed(base_seed, set_index, 99),
                beta=8.0,
            ).fit(memories.patterns)
            for factor in factors
        }
        for target_id in range(8):
            cue_seed = stable_seed(base_seed, set_index, target_id)
            # [输入] 同一 target 的四格条件读取同一个 25% 损坏 cue
            cue = c1_make_hamming_cue(memories.patterns[target_id], 0.25, cue_seed)
            for (fraction, similarity), model in models.items():
                final = model.retrieve(cue)
                rows.append({
                    "structure": "pairwise" if fraction == 0.0 else "mixed",
                    "similarity": similarity,
                    "pattern_set_id": set_index,
                    "target_id": target_id,
                    "top1_correct": e3_top1_memory(final, memories.patterns) == target_id,
                })
    return pd.DataFrame(rows)


h7_results = run_factorial_h7()
h7_results.to_json(
    artifact_root / "h7_factorial_raw.jsonl", orient="records", lines=True
)
h7_table = h7_results.groupby(["structure", "similarity"])["top1_correct"].mean().unstack()
p00 = h7_table.loc["pairwise", "dot"]
p01 = h7_table.loc["pairwise", "cosine"]
p10 = h7_table.loc["mixed", "dot"]
p11 = h7_table.loc["mixed", "cosine"]
# [观测·交互] 概率尺度 p11-p10-p01+p00；必须与四格原始率一起解释
interaction = p11 - p10 - p01 + p00
h7_table.plot(kind="bar", figsize=(7.4, 4.6), ylim=(0, 1), rot=0)
plt.ylabel("Top-1 identification rate")
plt.title("H7 complete 2×2 | structure × similarity")
plt.tight_layout()
plt.show()
display(Markdown(
    f"**本次运行的描述性结论**：概率尺度交互项为 `{interaction:+.3f}`。正值表示两项联合收益超过两个单独主效应的加和，负值表示抵消。\n\n"
    "**证据边界**：必须连同四格原始率一起解释；单个交互数不能证明通用协同，也不能替代论文的图像数据实验。"
))
display(h7_table.round(3))

## 实验 10：H4/H5 Curved/Explosive 的有效温度与迟滞

论文的核心状态反馈是 `beta_eff = beta / (1 + gamma*m²/2)`。这里复现对应的一维平均场动力学 `dm/dt = -m + tanh(beta_eff*m)`，并从低 beta 正向、从高 beta 反向延续稳定支，检验是否真的存在路径依赖。它是论文专属理论轨道，不与确定性 Top-1 主图混排。

实现公式对照本 notebook 内的函数；网络级随机 Glauber 实验可对照论文的 [official repository](https://github.com/MiguelAguilera/explosive-neural-networks)。

In [ ]:
# [机制] beta_eff=beta/(1+gamma*m²/2)，温度由当前 overlap 反馈调制
def effective_beta(beta: float, gamma: float, overlap: float) -> float:
    denominator = 1.0 + 0.5 * gamma * overlap**2
    # [失败边界] 分母非正时曲率参数进入非法域，返回 NaN 而不是继续画假曲线
    if denominator <= 0.0:
        return float("nan")
    return beta / denominator


# [动力学] 显式积分 dm/dt=-m+tanh(beta_eff*m)，并返回是否达到容差
def relax_curved_mean_field(
    initial_m: float, beta: float, gamma: float,
    *, dt: float = 0.05, tolerance: float = 1e-10, max_steps: int = 20_000,
) -> tuple[float, bool]:
    m = float(initial_m)
    for _ in range(max_steps):
        beta_eff = effective_beta(beta, gamma, m)
        if not np.isfinite(beta_eff):
            return float("nan"), False
        # [更新] 一步 Euler 积分；clip 只维护 overlap 的物理区间 [0,1]
        next_m = m + dt * (-m + np.tanh(beta_eff * m))
        next_m = float(np.clip(next_m, 0.0, 1.0))
        if abs(next_m - m) < tolerance:
            return next_m, True
        m = next_m
    return m, False


# [实验控制] 前一点的稳定解作为下一 beta 初值，保留扫描方向的路径依赖
def continuation_branch(beta_grid: np.ndarray, gamma: float, initial_m: float) -> pd.DataFrame:
    rows = []
    m = initial_m
    for beta in beta_grid:
        # [数值稳定] 给零支极小扰动，避免 gamma=0 因精确 m=0 产生伪迟滞
        if m < 1e-8:
            m = 1e-4
        m, converged = relax_curved_mean_field(m, float(beta), gamma)
        rows.append({"beta": beta, "gamma": gamma, "m": m, "converged": converged})
    return pd.DataFrame(rows)


# [机制对照] 记录参考反馈轨迹及它实际经历的 beta_eff 时间表
def feedback_trajectory(
    initial_m: float, beta: float, gamma: float,
    *, steps: int = 400, dt: float = 0.05,
) -> tuple[np.ndarray, np.ndarray]:
    overlaps = [float(initial_m)]
    beta_schedule = []
    for _ in range(steps):
        beta_eff = effective_beta(beta, gamma, overlaps[-1])
        beta_schedule.append(beta_eff)
        next_m = overlaps[-1] + dt * (
            -overlaps[-1] + np.tanh(beta_eff * overlaps[-1])
        )
        overlaps.append(float(np.clip(next_m, 0.0, 1.0)))
    return np.asarray(overlaps), np.asarray(beta_schedule)


# [反事实] 对扰动初态播放同一时间表，但不允许 beta 根据新状态反馈
def open_loop_trajectory(
    initial_m: float, beta_schedule: np.ndarray, *, dt: float = 0.05
) -> np.ndarray:
    overlaps = [float(initial_m)]
    for beta_eff in beta_schedule:
        next_m = overlaps[-1] + dt * (
            -overlaps[-1] + np.tanh(beta_eff * overlaps[-1])
        )
        overlaps.append(float(np.clip(next_m, 0.0, 1.0)))
    return np.asarray(overlaps)


beta_grid = np.linspace(0.2, 3.0, 120)
gamma_values = (0.0, -0.5, -1.0, -1.5)
branches = []
# [实验控制] gamma=0 是固定温度基线，负 gamma 是爆发式正反馈条件
for gamma in gamma_values:
    forward = continuation_branch(beta_grid, gamma, initial_m=1e-6).assign(direction="forward")
    backward = continuation_branch(beta_grid[::-1], gamma, initial_m=0.999).assign(direction="backward")
    branches.extend([forward, backward])
curved_branches = pd.concat(branches, ignore_index=True)

# [实验控制] reference 与 perturbed-feedback/matched-open-loop 构成 H4 三组对照
reference_feedback, matched_schedule = feedback_trajectory(
    initial_m=0.85, beta=0.8, gamma=-1.5
)
perturbed_feedback, _ = feedback_trajectory(
    initial_m=0.35, beta=0.8, gamma=-1.5
)
perturbed_open_loop = open_loop_trajectory(0.35, matched_schedule)

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.4))
for (gamma, direction), group in curved_branches.groupby(["gamma", "direction"]):
    axes[0].plot(
        group["beta"], group["m"],
        linestyle="-" if direction == "forward" else "--",
        label=f"gamma={gamma:g}, {direction}",
    )
overlap_grid = np.linspace(0, 1, 200)
for gamma in gamma_values:
    axes[1].plot(
        overlap_grid,
        [effective_beta(1.0, gamma, m) for m in overlap_grid],
        label=f"gamma={gamma:g}",
    )
axes[2].plot(reference_feedback, label="feedback, reference initial")
axes[2].plot(perturbed_feedback, label="feedback, perturbed initial")
axes[2].plot(perturbed_open_loop, "--", label="matched open-loop schedule")
axes[0].set(xlabel="Control beta", ylabel="Stable overlap m", title="H5 forward/backward continuation")
axes[1].set(xlabel="Overlap m", ylabel="Effective beta (base beta=1)", title="H4 state-feedback temperature")
axes[2].set(xlabel="Integration step", ylabel="Overlap m", title="H4 feedback vs matched schedule")
axes[0].legend(fontsize=7, ncol=2)
axes[1].legend(fontsize=8)
axes[2].legend(fontsize=7)
fig.tight_layout()
plt.show()

wide = curved_branches.pivot_table(index=["gamma", "beta"], columns="direction", values="m").reset_index()
# [观测·数据] 同一 beta 的正反稳定支差值，是 H5 迟滞的直接量尺
wide["branch_gap"] = (wide["forward"] - wide["backward"]).abs()
gap_summary = wide.groupby("gamma", as_index=False)["branch_gap"].max()
curved_branches.to_json(
    artifact_root / "h45_curved_branches.jsonl", orient="records", lines=True
)
# [存储] 保存三条时间轨迹，图不能成为唯一证据
h4_counterfactual = pd.DataFrame({
    "step": np.arange(len(reference_feedback)),
    "reference_feedback": reference_feedback,
    "perturbed_feedback": perturbed_feedback,
    "perturbed_matched_open_loop": perturbed_open_loop,
})
h4_counterfactual.to_json(
    artifact_root / "h4_feedback_counterfactual.jsonl",
    orient="records", lines=True,
)
display(Markdown(
    "**本次运行的描述性结论**\n\n"
    + "\n".join(
        f"- gamma={row.gamma:g}：正反扫描最大分支差={row.branch_gap:.3f}。"
        for row in gap_summary.itertuples(index=False)
    )
    + "\n\n**证据边界**：平均场迟滞是机制证据，不等同于有限 N 随机网络中的统计显著性；未收敛点必须先排查，不能当作相变。"
))
display(gap_summary.round(3))

## 实验 11：H6 PSHN 分组数与 feature-to-prototype

下面给出论文尺度的 MNIST 入口：`M=10,000`，`k∈{7,16,28,112}`，右侧遮挡 `theta∈{0,.25,.35}`，10 次不同子采样。除了论文的“完美恢复数量”，还记录输出离目标实例和目标类别原型的误差，防止高阶模型输出原型却被单一指标掩盖。

默认 `RUN_PSHN_MNIST=False`，因为这是昂贵、会下载 MNIST 的论文专属实验；切换为 `True` 后才在 Colab 执行。关闭时不会生成伪结果。

In [ ]:
# [执行门] 默认关闭，确保普通 Run All 不会意外下载数据或启动论文尺度计算
RUN_PSHN_MNIST = False


# [实验控制] 论文尺度 M、k、遮挡率、10 次子采样和批大小集中登记
@dataclass(frozen=True)
class PSHNMNISTConfig:
    memory_count: int = 10_000
    query_count: int = 10_000
    groups: tuple[int, ...] = (7, 16, 28, 112)
    occlusion_levels: tuple[float, ...] = (0.0, 0.25, 0.35)
    trials: int = 10
    query_batch_size: int = 32
    base_seed: int = 20260909


# [更新] 论文 PSHN 一次读出的批量版；输出仍是 {-1,+1}
def pshn_batch_update(
    queries: torch.Tensor, memories: torch.Tensor, groups: int
) -> torch.Tensor:
    M, N = memories.shape
    batch = len(queries)
    group_size = N // groups
    X = memories.reshape(M, groups, group_size)
    Q = queries.reshape(batch, groups, group_size)
    # [中介变量] (B,k,N/k) 与 (M,k,N/k)→(B,M,k) 分组相关度
    correlations = torch.einsum("bkg,Mkg->bMk", Q, X)
    # [数值稳定] k=112 时直接连乘会溢出，改在符号/对数域构造等价相对系数
    nonzero = correlations != 0
    safe_sign = torch.where(nonzero, torch.sign(correlations), torch.ones_like(correlations))
    safe_log_abs = torch.where(
        nonzero, torch.log(correlations.abs()), torch.zeros_like(correlations)
    )
    # [中介变量] 排除第 g 组后仍含零相关度，则该 memory-group 系数必须为零
    zero_count = (~nonzero).sum(dim=2, keepdim=True)
    valid = zero_count - (~nonzero).to(torch.int64) == 0
    sign_without = safe_sign.prod(dim=2, keepdim=True) * safe_sign
    log_without = safe_log_abs.sum(dim=2, keepdim=True) - safe_log_abs
    masked_log = torch.where(valid, log_without, torch.full_like(log_without, -torch.inf))
    # [数值稳定] 每条 query 减去共同最大 log 系数；只乘正比例常数，不改变局部场符号
    scale = masked_log.amax(dim=(1, 2), keepdim=True)
    scale = torch.where(torch.isfinite(scale), scale, torch.zeros_like(scale))
    C = torch.where(valid, sign_without * torch.exp(log_without - scale), 0.0)
    # [更新] C:(B,M,k) 与 X:(M,k,N/k)→fields:(B,k,N/k)
    fields = torch.einsum("bMk,Mkg->bkg", C, X)
    output = torch.sign(fields).reshape(batch, N)
    ties = output == 0
    output[ties] = queries[ties]
    return output


# [输入] 按论文把图像右侧连续 100*theta% 像素设为 -1
def right_occlusion(patterns: torch.Tensor, theta: float) -> torch.Tensor:
    images = patterns.reshape(-1, 28, 28).clone()
    width = int(round(28 * theta))
    if width:
        images[:, :, 28 - width:] = -1
    return images.reshape(-1, 784)


# [执行] 每个 trial 从完整 70K MNIST 中重新无放回抽取 10K 记忆
def run_pshn_mnist(config: PSHNMNISTConfig) -> pd.DataFrame:
    from torchvision.datasets import MNIST

    # [环境] 论文尺度优先使用 Colab GPU；CPU 仍可运行但会很慢
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train = MNIST("/content/data", train=True, download=True)
    test = MNIST("/content/data", train=False, download=True)
    pixels = torch.cat([train.data, test.data]).reshape(-1, 784)
    labels = torch.cat([train.targets, test.targets])
    # [输入] 像素阈值化为 {-1,+1}，与论文实验编码一致
    binary = torch.where(pixels >= 128, 1.0, -1.0).to(torch.float32)
    rows = []
    for trial in range(config.trials):
        generator = make_generator(stable_seed(config.base_seed, trial))
        # [实验控制] trial 改变 10K 子样本；同一 trial 的全部 k/theta 共享样本
        indices = torch.randperm(len(binary), generator=generator)[:config.memory_count]
        memories = binary[indices].to(device)
        memory_labels = labels[indices].to(device)
        query_count = min(config.query_count, config.memory_count)
        targets = memories[:query_count]
        target_labels = memory_labels[:query_count]
        # [中介变量] 每个数字类别的符号均值原型，用于 feature-to-prototype 诊断
        prototypes = torch.stack([
            torch.sign(memories[memory_labels == digit].mean(dim=0))
            for digit in range(10)
        ])
        for groups in config.groups:
            if 784 % groups:
                raise ValueError(f"groups={groups} does not divide 784")
            for theta in config.occlusion_levels:
                # [输入] 遮挡在 group 循环内固定生成；同一 k 的 query 顺序不变
                queries = right_occlusion(targets, theta)
                for start in range(0, query_count, config.query_batch_size):
                    stop = min(start + config.query_batch_size, query_count)
                    output = pshn_batch_update(queries[start:stop], memories, groups)
                    batch_targets = targets[start:stop]
                    batch_labels = target_labels[start:stop]
                    # [观测·数据] exact 要求 784 位全对，严格复核论文完美恢复量
                    exact = torch.all(output == batch_targets, dim=1)
                    instance_error = (output != batch_targets).float().mean(dim=1)
                    # [观测·机制] 与类别原型误差和实例误差并列，防止原型化输出被算成容量收益
                    prototype_error = (
                        output != prototypes[batch_labels]
                    ).float().mean(dim=1)
                    # [观测·错误] 原型最近类别用于识别跨类别误检，不能只看平均像素误差
                    prototype_prediction = torch.argmax(output @ prototypes.T, dim=1)
                    for offset in range(stop - start):
                        rows.append({
                            "trial": trial,
                            "groups": groups,
                            "occlusion": theta,
                            "query_id": start + offset,
                            "exact_recall": bool(exact[offset].item()),
                            "instance_error": float(instance_error[offset].item()),
                            "prototype_error": float(prototype_error[offset].item()),
                            "prototype_class_correct": bool(
                                prototype_prediction[offset] == batch_labels[offset]
                            ),
                        })
    return pd.DataFrame(rows)


# [执行门] 只有用户在 Colab 显式打开开关，下面才下载并产生 H6 结果
if RUN_PSHN_MNIST:
    pshn_mnist = run_pshn_mnist(PSHNMNISTConfig())
    pshn_mnist.to_json(
        artifact_root / "h6_pshn_mnist_raw.jsonl", orient="records", lines=True
    )
    pshn_summary = pshn_mnist.groupby(["groups", "occlusion"], as_index=False).agg(
        exact_recall=("exact_recall", "mean"),
        instance_error=("instance_error", "mean"),
        prototype_error=("prototype_error", "mean"),
        prototype_class_correct=("prototype_class_correct", "mean"),
    )
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.4))
    for theta, group in pshn_summary.groupby("occlusion"):
        axes[0].plot(group["groups"], group["exact_recall"], marker="o", label=f"theta={theta:g}")
        axes[1].plot(
            group["groups"], group["instance_error"] - group["prototype_error"],
            marker="o", label=f"theta={theta:g}",
        )
    axes[0].set(xlabel="PSHN groups k", ylabel="Perfect recovery rate", title="H6 MNIST instance recovery")
    axes[1].set(
        xlabel="PSHN groups k", ylabel="Instance error - prototype error",
        title="H6 feature-to-prototype diagnostic",
    )
    for axis in axes:
        axis.set_xscale("log", base=2)
        axis.legend()
    fig.tight_layout()
    plt.show()
    display(Markdown(
        "**本次运行的描述性结论**：左图复核完美恢复，右图为正时表示输出比目标实例更靠近类别原型。\n\n"
        "**证据边界**：必须同时检查类别正确率与同类误检；不同 k 的改善不能单独归因为容量。"
    ))
    display(pshn_summary.round(4))
else:
    display(Markdown(
        "**H6 尚未执行**：将 `RUN_PSHN_MNIST` 改为 `True` 后，Colab 才会下载 MNIST 并运行论文尺度实验；当前没有图，也没有数值结论。"
    ))

## 12. 读图顺序与结论门

1. 先看 U2/U3/U4 的共同 Top-1 曲线，确认相同数据、线索和 trial 配对。
2. 再看 U1 固定点与 U5/U6 动力学；不适用的模型不进入对应排名。
3. 用 U7 检查 native 资源差异，再用 H1–H3 的同参数消融判断高阶结构收益。
4. H4–H7 只解释机制，不把论文专属数字混入统一冠军榜。

只有当原始记录、失败行、随机种子、预算与图下结论一一对应时，才允许写“本次运行支持”。有限扫描不支持线性、多项式、指数或双指数容量的渐近声明。